# Tutorial 3 — Merging multiple metabolomics assays

Metabolomics studies may include several assay tables generated using
different analytical platforms or acquisition modes.

In this tutorial, we merge the three assay tables from the MTBLS1866
study into a single study-level feature table.

The merge procedure:

1. reads the sample identifiers from the study metadata,
2. matches sample columns across all assays,
3. reorders the columns according to the metadata,
4. stacks all metabolite features vertically,
5. exports a single merged abundance table.

No preprocessing, statistical analysis, metabolite mapping, or pathway
analysis is performed in this tutorial.

## 1. Installation and imports

In [2]:
!git clone -q https://github.com/tiganouri/ChemEquivMapper.git

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Define input files

In [4]:
REPO = Path("/content/ChemEquivMapper")
DATA_DIR = REPO / "experimental_data"

assay_paths = {
    "MTBLS1866_GCxGC": (
        DATA_DIR
        / "m_MTBLS1866_GCxGC-MS_positive__metabolite_profiling_v2_maf.txt"
    ),
    "MTBLS1866_LCpos_RP": (
        DATA_DIR
        / "m_MTBLS1866_LC-MS_positive_reverse-phase_metabolite_profiling_v2_maf.txt"
    ),
    "MTBLS1866_LCneg_RP": (
        DATA_DIR
        / "m_MTBLS1866_LC-MS_negative_reverse-phase_metabolite_profiling_v2_maf.txt"
    ),
}

metadata_path = (
    DATA_DIR
    / "metadata_group_MTBLS1866.txt"
)

OUTPUT_DIR = REPO / "derived" / "merged_inputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "MERGED_MTBLS1866.txt"

pd.DataFrame(
    {
        "assay": assay_paths.keys(),
        "file": [path.name for path in assay_paths.values()],
        "exists": [path.exists() for path in assay_paths.values()],
    }
)

,assay,file,exists
0,MTBLS1866_GCxGC,m_MTBLS1866_GCxGC-MS_positive__metabolite_prof...,True
1,MTBLS1866_LCpos_RP,m_MTBLS1866_LC-MS_positive_reverse-phase_metab...,True
2,MTBLS1866_LCneg_RP,m_MTBLS1866_LC-MS_negative_reverse-phase_metab...,True


# 3. Read the assay and metadata tables

In [10]:
def read_table_auto(path: Path, sep: str = "\t") -> pd.DataFrame:
    """Read a tab-separated table using common text encodings."""

    encodings = [
        "utf-8",
        "utf-8-sig",
        "utf-16",
        "latin-1",
        "cp1252",
    ]

    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                sep=sep,
                encoding=encoding,
                dtype=str,
            )
        except UnicodeDecodeError as error:
            last_error = error

    raise RuntimeError(
        f"Could not decode file: {path}"
    ) from last_error

In [11]:
def read_feature_table(path: Path) -> pd.DataFrame:
    """
    Read a metabolomics feature table.

    The first column is treated as the metabolite or feature identifier.
    All remaining columns are treated as sample abundance columns.
    """

    df = read_table_auto(path)

    if df.shape[1] < 2:
        raise ValueError(
            f"{path.name} does not look like a feature table."
        )

    feature_column = df.columns[0]

    df = df.rename(
        columns={feature_column: "metabolite_identification"}
    )

    df["metabolite_identification"] = (
        df["metabolite_identification"]
        .astype(str)
        .str.strip()
    )

    for column in df.columns[1:]:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    return df.set_index("metabolite_identification")


def read_metadata_samples(metadata_path: Path) -> list[str]:
    """Extract sample identifiers from the study metadata."""

    metadata = read_table_auto(metadata_path)

    sample_column_candidates = [
        "sample",
        "Sample",
        "SampleID",
        "sample_id",
        "sample_name",
        "run",
        "Run",
    ]

    sample_column = next(
        (
            column
            for column in sample_column_candidates
            if column in metadata.columns
        ),
        metadata.columns[0],
    )

    samples = (
        metadata[sample_column]
        .dropna()
        .astype(str)
        .str.strip()
    )

    samples = [
        sample
        for sample in samples
        if sample and sample.lower() != "nan"
    ]

    if len(samples) != len(set(samples)):
        raise ValueError(
            "Duplicate sample identifiers were found in the metadata."
        )

    return samples

In [8]:
def merge_assays(
    assay_paths: dict[str, Path],
    metadata_path: Path,
    output_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Align and vertically concatenate multiple assay tables.

    Sample columns are ordered according to the metadata file.
    Missing assay measurements are retained as NaN.
    """

    metadata_samples = read_metadata_samples(metadata_path)

    assay_blocks = []
    merge_log = []

    for assay_name, assay_path in assay_paths.items():

        assay = read_feature_table(assay_path)

        assay_samples = list(assay.columns)

        matched_samples = [
            sample
            for sample in metadata_samples
            if sample in assay_samples
        ]

        missing_samples = [
            sample
            for sample in metadata_samples
            if sample not in assay_samples
        ]

        extra_samples = [
            sample
            for sample in assay_samples
            if sample not in metadata_samples
        ]

        # Reorder the assay columns according to the metadata.
        # Samples absent from an assay are introduced as NaN columns.
        assay = assay.reindex(columns=metadata_samples)

        assay.insert(0, "source_assay", assay_name)

        assay_blocks.append(assay)

        merge_log.append(
            {
                "assay": assay_name,
                "file": assay_path.name,
                "n_features": assay.shape[0],
                "n_metadata_samples": len(metadata_samples),
                "n_matched_samples": len(matched_samples),
                "n_missing_samples": len(missing_samples),
                "n_extra_samples": len(extra_samples),
                "missing_samples": " | ".join(missing_samples),
                "extra_samples": " | ".join(extra_samples),
            }
        )

    merged = pd.concat(
        assay_blocks,
        axis=0,
        join="outer",
        sort=False,
    )

    merged = merged.reset_index()

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    merged.to_csv(
        output_path,
        sep="\t",
        index=False,
    )

    merge_report = pd.DataFrame(merge_log)

    return merged, merge_report

In [12]:
merged_table, merge_report = merge_assays(
    assay_paths=assay_paths,
    metadata_path=metadata_path,
    output_path=output_path,
)

print(f"Merged table saved to: {output_path}")
print(f"Number of assay tables: {len(assay_paths)}")
print(f"Number of merged features: {merged_table.shape[0]:,}")
print(f"Number of metadata samples: {merged_table.shape[1] - 2:,}")

display(merge_report)

Merged table saved to: /content/ChemEquivMapper/derived/merged_inputs/MERGED_MTBLS1866.txt
Number of assay tables: 3
Number of merged features: 809
Number of metadata samples: 157


,assay,file,n_features,n_metadata_samples,n_matched_samples,n_missing_samples,n_extra_samples,missing_samples,extra_samples
0,MTBLS1866_GCxGC,m_MTBLS1866_GCxGC-MS_positive__metabolite_prof...,251,157,127,30,30,2 POS | CM POS | IRA POS | 3 POS | RI POS | SG...,IRA | CM | CE | SG | RI | POS 2 | POS 3 | A | ...
1,MTBLS1866_LCpos_RP,m_MTBLS1866_LC-MS_positive_reverse-phase_metab...,469,157,127,30,28,13 | 2 POS | CM POS | IRA POS | 3 POS | RI POS...,SG | CE | IRA | CM | RI | DLMC 93 | POS 2 | PO...
2,MTBLS1866_LCneg_RP,m_MTBLS1866_LC-MS_negative_reverse-phase_metab...,89,157,127,30,28,13 | 2 POS | CM POS | IRA POS | 3 POS | RI POS...,RI | POS 2 | POS 3 | DLMC 93 | CE | CM | IRA |...


In [13]:
report_path = OUTPUT_DIR / "MTBLS1866_merge_report.tsv"

merge_report.to_csv(
    report_path,
    sep="\t",
    index=False,
)

display(merged_table.head())

print(f"Merged feature table: {output_path}")
print(f"Merge report: {report_path}")

,metabolite_identification,source_assay,1,41,101,62,104,66,106,67,...,26 NEG,28 NEG,29 NEG,30 NEG,32 NEG,35 NEG,37 NEG,46 NEG,5 NEG,6 NEG
0,O-methoxycarbonyl-(9H-fluoren-9-yl)methanol,MTBLS1866_GCxGC,1.137101,0.898727,2.018895,1.811399,1.629445,1.643451,1.724833,1.102475,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"(R*,S*)-2,3-Dihydroxybutanoic acid",MTBLS1866_GCxGC,1.923148,7.848772,20.291801,4.976251,6.247156,25.285306,21.222278,6.072143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(Z)-Docos-9-enenitrile,MTBLS1866_GCxGC,1.376149,0.642911,1.748186,1.315680,1.601515,1.444794,2.209807,1.757660,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"1-(2,5-Dimethoxyphenyl)-1-[(trimethylsilyl)oxy...",MTBLS1866_GCxGC,0.422232,0.576274,0.000000,0.222975,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"1,2-Benzenedicarboxylic acid, bis(2-methylprop...",MTBLS1866_GCxGC,0.415875,0.312073,1.062138,0.551296,0.716869,0.405624,0.902120,0.535100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Merged feature table: /content/ChemEquivMapper/derived/merged_inputs/MERGED_MTBLS1866.txt
Merge report: /content/ChemEquivMapper/derived/merged_inputs/MTBLS1866_merge_report.tsv
